In [ ]:
import numpy as np
import matplotlib.pyplot as plt


import plot_func
import FMfuncs
import torch


import configs
import dataset
import PGFM
import FM2 #i.e., FMDD

In [ ]:
1

In [ ]:
data = np.load("./data/projected_10DGaussian.npy")
data = torch.tensor(data, dtype=torch.float32, device = configs.device)

In [ ]:
FMclass = FMfuncs.OTFlowMatching()
PGFMclass = PGFM.PGFM()
FM2class =FM2.FM2()

In [ ]:
FMclass.train(data)

In [ ]:
# PGFMclass.train2_2stage(data, './saved_model/FM_subspace_500000.pth')
FM2class.train2_2stage(data, './saved_model/FM_subspace_500000.pth')

In [ ]:
PGFMclass.train2_2stage(data, './saved_model/FM_subspace_500000.pth')

In [ ]:
ckpt1 = torch.load('./saved_model/FM_subspace_500000.pth', map_location=configs.device, weights_only=True)
# ckpt2 = torch.load('./saved_model/PGFM_subspace_20000.pth', map_location=configs.device, weights_only=True)
ckpt2 = torch.load('./saved_model/FM2_subspace_20000.pth', map_location=configs.device, weights_only=True)
# PGFM_subspace_Mar27

In [ ]:
PGFMclass = FM2class

stage1model = PGFMclass.get_untrained_model()
stage1model.load_state_dict(ckpt1)
stage2model = PGFMclass.policy
stage2model.load_state_dict(ckpt2)
res = PGFMclass.PGFMsample_train2(stage1model, stage2model, 10000)


mycoeff = np.ones(configs.d_model+1)
mycoeff[-1] = 10
print(dataset.distance_data_to_plane(res.cpu().numpy(), mycoeff))
print(plot_func.compute_SWD(data[:10000].cpu().numpy(), res.cpu().numpy()))

In [ ]:
stage1model = FMclass.get_untrained_model()
stage1model.load_state_dict(ckpt1)


res = FMfuncs.sampler(stage1model, 10000, stoptime=1)
mycoeff = np.ones(configs.d_model+1)
mycoeff[-1] = 10
print(dataset.distance_data_to_plane(res.cpu().numpy(), mycoeff))
print(plot_func.compute_SWD(data[:10000].cpu().numpy(), res.cpu().numpy()))

In [ ]:
tolerance = 5e-4
n_trails = 100
avg_distance_FM = np.zeros(n_trails)
avg_distance_PGFM = np.zeros(n_trails)
SWD_FM = np.zeros(n_trails)
SWD_PGFM = np.zeros(n_trails)
invalid_num_FM = np.zeros(n_trails)
invalid_num_PGFM = np.zeros(n_trails)
mycoeff = np.ones(configs.d_model+1)
mycoeff[-1] = 10

for i in range(n_trails):
    if (i+1)%10==0:
        print(i+1)
    random_index = np.random.choice(data.shape[0], 10000, replace=False)
    resPGFM = PGFMclass.PGFMsample_train2(stage1model, stage2model, 10000)
    resFM = FMfuncs.sampler(stage1model, 10000, stoptime=1)
    
    distancePGFM_batch, distancePGFM = dataset.distance_data_to_plane(resPGFM.cpu().numpy(), mycoeff)
    distanceFM_batch, distanceFM = dataset.distance_data_to_plane(resFM.cpu().numpy(), mycoeff)
    
    invalid_num_FM[i] = np.sum(distanceFM_batch > tolerance)
    invalid_num_PGFM[i] = np.sum(distancePGFM_batch > tolerance)
    
    avg_distance_PGFM[i] = distancePGFM
    avg_distance_FM[i] = distanceFM

    SWDtmpPGFM = plot_func.compute_SWD(data[random_index].cpu().numpy(), resPGFM.cpu().numpy())
    SWDtmpFM = plot_func.compute_SWD(data[random_index].cpu().numpy(), resFM.cpu().numpy())
    SWD_FM[i] = SWDtmpFM
    SWD_PGFM[i] = SWDtmpPGFM





In [ ]:
print(" FM SWD: ", f"{np.mean( SWD_FM):.7f}", r"\pm", f"{np.std( SWD_FM):.7f}")
print(" FM num out: ", f"{np.mean( invalid_num_FM):.7f}", r"\pm", f"{np.std( invalid_num_FM):.7f}")
print("FM avg distance: ", f"{np.mean(avg_distance_FM):.7f}", r"\pm", f"{np.std(avg_distance_FM):.7f}")

print("PGFM SWD: ", f"{np.mean(SWD_PGFM):.4f}", r"\pm", f"{np.std(SWD_PGFM):.4f}")
print("PGFM num out: ", f"{np.mean(invalid_num_PGFM):.4f}", r"\pm", f"{np.std(invalid_num_PGFM):.4f}")
print("PGFM avg distance: ", f"{np.mean(avg_distance_PGFM):.7f}", r"\pm", f"{np.std(avg_distance_PGFM):.7f}")

In [ ]:
distanceFM